In [1]:
from dotenv import load_dotenv
import json
import whisper
from pyannote.audio import Pipeline
from pyannote_whisper.utils import diarize_text
import os

In [2]:
file = 'data/audio_sample.mp3'

In [3]:
load_dotenv('openai.env')

True

In [8]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                    use_auth_token=os.getenv('PYANNOTE'))

config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/399 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/221 [00:00<?, ?B/s]

In [ ]:
diarization_result = pipeline(file, num_speakers=2)

In [5]:
model = whisper.load_model("base")

In [6]:
asr_result = model.transcribe(file)

/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [11]:
final_result = diarize_text(asr_result, diarization_result)

In [38]:
speaker_matches = {list(final_result[i])[0].start : list(final_result[i])[1] for i in range(0, len(final_result))}

In [44]:
switches = list(speaker_matches.keys())
for i in range(0, len(asr_result['segments'])):
    curr_start = asr_result['segments'][i]['start']
    for j in range(0, len(switches)):
        if curr_start < switches[j]:
            switch = switches[j-1]
            break
    asr_result['segments'][i]['speaker'] = speaker_matches[switch]

In [47]:
result = asr_result

In [48]:
with open('transcript.json', 'w') as out_path:  
    json.dump(result, out_path)